In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

from langgraph.checkpoint.memory import InMemorySaver #This is used for persistence. This stores the state in RAM.

In [3]:
load_dotenv()

model = ChatGroq(model="llama-3.3-70b-versatile")

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def create_joke(state: JokeState):
    topic = state['topic']
    prompt = f'generate a joke on the topic {topic}'
    response = model.invoke(prompt).content

    return {"joke": response}

In [6]:
def explain(state: JokeState):
    joke = state['joke']
    prompt = f"Explain the joke: \n {joke}"
    response = model.invoke(prompt).content

    return {"explanation": response}

In [10]:
graph = StateGraph(JokeState)

graph.add_node("create_joke", create_joke)
graph.add_node("explain", explain)

graph.add_edge(START, "create_joke")
graph.add_edge("create_joke","explain")
graph.add_edge("explain",END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)


In [11]:
config1 = {"configurable": {"thread_id": "1"}}       
response = workflow.invoke({"topic":"pizza"}, config1)

In [ ]:
workflow.get_state(config1) #Gets the final state of the thread_id

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke is funny because it uses a pun to create a wordplay between the literal meaning of "crusty" (referring to the crust of a pizza) and the figurative meaning of "crusty" (meaning grumpy or irritable).\n\nIn this joke, the setup "Why was the pizza in a bad mood?" primes the listener to expect a reason for the pizza\'s bad mood. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a way that is both literal (the pizza has a crust) and figurative (the pizza is feeling grumpy). This unexpected twist creates the humor.\n\nIn essence, the joke is saying that the pizza is in a bad mood because it\'s feeling a little grumpy, but it\'s using a clever play on words to make the connection between the pizza\'s crust and its emotional state.'}, next=(), config={'configur

In [ ]:
list(workflow.get_state_history(config1)) #Gets the entire state history corresponding to the thread_id

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'A classic play on words. The joke is funny because it uses a pun to create a wordplay between the literal meaning of "crusty" (referring to the crust of a pizza) and the figurative meaning of "crusty" (meaning grumpy or irritable).\n\nIn this joke, the setup "Why was the pizza in a bad mood?" primes the listener to expect a reason for the pizza\'s bad mood. The punchline "Because it was feeling a little crusty" subverts this expectation by using the word "crusty" in a way that is both literal (the pizza has a crust) and figurative (the pizza is feeling grumpy). This unexpected twist creates the humor.\n\nIn essence, the joke is saying that the pizza is in a bad mood because it\'s feeling a little grumpy, but it\'s using a clever play on words to make the connection between the pizza\'s crust and its emotional state.'}, next=(), config={'configu